# NYC 311 SODA2 API connection test

Run the cells in order when you are ready. **This notebook has not been executed against the API.**
Select the repository's `.venv/bin/python` kernel, or launch JupyterLab from the repository root with `.venv/bin/jupyter lab`.

This connection test loads a bounded sample into **`df`**, using your selected fields and `created_date > 2024-10-18T10:17:14` filter. It defaults to 1,000 rows, not the entire matching dataset. The selected fields include detailed locations; records stay in memory and the preview uses aggregate-level fields. Clear all outputs before committing the notebook.


In [1]:
import os
import sys
import time
from datetime import UTC, datetime
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

# Locate this repository whether Jupyter starts at its root or notebook folder.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "docs/PROJECT_PLAN.md").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from within the project repository.")
load_dotenv(PROJECT_ROOT / ".env", override=False)
print(f"Python: {sys.version.split()[0]}")
print(f"Kernel: {sys.executable}")


Python: 3.13.5
Kernel: /Users/onurerbey/Library/Mobile Documents/com~apple~CloudDocs/GitHub/nyc-311-data-vault-dbt/.venv/bin/python


## Configure the query

The endpoint remains SODA2 `/resource/erm2-nwe9.json`. An optional `SOCRATA_APP_TOKEN` is read from the environment or local `.env` and sent in the `X-App-Token` header.
The query preserves your 44 selected fields, cutoff and descending date ordering. `unique_key` breaks date ties; `LIMIT` bounds this connection test. Change `ROW_LIMIT` below to adjust the sample (1–5,000).

This is a standalone exploratory query, outside the planned DEP/DSNY January–June 2026 warehouse cohort. No warehouse tables or files are written.


In [2]:
API_ENDPOINT = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
CREATED_AFTER = "2024-10-18T10:17:14"
ROW_LIMIT = 1_000
if not isinstance(ROW_LIMIT, int) or not 1 <= ROW_LIMIT <= 5_000:
    raise ValueError("ROW_LIMIT must be an integer from 1 to 5,000.")

SELECTED_FIELDS = [
    "unique_key", "created_date", "closed_date", "agency", "agency_name",
    "complaint_type", "descriptor", "descriptor_2", "location_type",
    "incident_zip", "incident_address", "street_name", "cross_street_1",
    "cross_street_2", "intersection_street_1", "intersection_street_2",
    "address_type", "city", "landmark", "facility_type", "status", "due_date",
    "resolution_description", "resolution_action_updated_date", "community_board",
    "council_district", "police_precinct", "bbl", "borough",
    "x_coordinate_state_plane", "y_coordinate_state_plane", "open_data_channel_type",
    "park_facility_name", "park_borough", "vehicle_type", "taxi_company_borough",
    "taxi_pick_up_location", "bridge_highway_name", "bridge_highway_direction",
    "road_ramp", "bridge_highway_segment", "latitude", "longitude", "location",
]
select_clause = ", ".join(f"`{field}`" for field in SELECTED_FIELDS)
QUERY = (
    f"SELECT {select_clause} "
    f'WHERE `created_date` > "{CREATED_AFTER}" :: floating_timestamp '
    "ORDER BY `created_date` DESC NULL FIRST, `unique_key` DESC "
    f"LIMIT {ROW_LIMIT}"
)
headers = {"Accept": "application/json"}
if token := os.getenv("SOCRATA_APP_TOKEN"):
    headers["X-App-Token"] = token
print(f"Ready to request up to {ROW_LIMIT:,} rows; no request sent yet.")


Ready to request up to 1,000 rows; no request sent yet.


## Define the connection test

This function uses URL-encoded `$query`, explicit timeouts, and up to three attempts for timeouts, transport errors, HTTP 429 and server errors. Backoff is bounded; a longer `Retry-After` stops the test so you can retry later. Other HTTP errors (including invalid field names) fail immediately. Live field availability is checked only when you run the request; there is no silent column substitution.


In [3]:
def fetch_records(client, query):
    for attempt in range(3):
        try:
            response = client.get(API_ENDPOINT, params={"$query": query})
        except httpx.TransportError:
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)
            continue

        if (response.status_code == 429 or 500 <= response.status_code < 600) and attempt < 2:
            retry_after = response.headers.get("Retry-After")
            if retry_after is not None:
                try:
                    delay = float(retry_after)
                except ValueError:
                    raise RuntimeError("Server requested a later retry; rerun later.") from None
                if not 0 <= delay <= 30:
                    raise RuntimeError("Server requested a later retry; rerun later.")
            else:
                delay = 2 ** attempt
            time.sleep(delay)
            continue
        if response.is_error:
            # Include only the API's error message, never request headers/tokens.
            try:
                message = response.json().get("message", "Request failed")
            except (ValueError, AttributeError):
                message = "Request failed; inspect the HTTP status and endpoint."
            raise RuntimeError(f"SODA2 HTTP {response.status_code}: {message}")
        response.raise_for_status()
        records = response.json()
        if not isinstance(records, list) or any(not isinstance(row, dict) for row in records):
            raise ValueError("Expected a JSON array of request objects.")
        return records


## Run this cell to call the API and load `df`

This is the **first cell that sends a network request**. A successful response creates `df` with the selected source columns (including entirely missing/null columns). IDs, ZIP codes and source timestamps retain their source representation. Capture times are kept separately in `capture_metadata`; they are observation times, not source event times or Vault load times.


In [4]:
# Clear a previous result so a failed rerun cannot leave stale data visible as current.
df = None
capture_metadata = None
capture_started_at_utc = datetime.now(UTC)
with httpx.Client(headers=headers, timeout=httpx.Timeout(60.0, connect=10.0)) as client:
    records = fetch_records(client, QUERY)
capture_finished_at_utc = datetime.now(UTC)

if len(records) > ROW_LIMIT:
    raise ValueError("API returned more than the configured row limit.")
df = pd.DataFrame.from_records(records, columns=SELECTED_FIELDS)
if df["unique_key"].isna().any() or df["unique_key"].duplicated().any():
    raise ValueError("Response contains missing or duplicate request identifiers.")
capture_metadata = {
    "record_source": "NYC311:erm2-nwe9",
    "capture_started_at_utc": capture_started_at_utc.isoformat(),
    "snapshot_at_utc": capture_finished_at_utc.isoformat(),
    "query": QUERY,
    "row_limit": ROW_LIMIT,
    "fetched_rows": len(df),
    "scope": "bounded connection-test sample; not a complete cohort snapshot",
}
df.attrs.update(capture_metadata)
del records
print(f"Loaded {len(df):,} rows and {len(df.columns)} source columns into df.")
if df.empty:
    print("Connection succeeded; the query returned no matching records.")
elif len(df) == ROW_LIMIT:
    print("Sample limit reached; additional matching records may exist.")


Loaded 1,000 rows and 44 source columns into df.
Sample limit reached; additional matching records may exist.


In [5]:
# Compact preview without addresses, free-text descriptions or precise coordinates.
df[["unique_key", "created_date", "agency", "complaint_type", "status", "borough"]].head()


,unique_key,created_date,agency,complaint_type,status,borough
0,70774279,2026-10-17T02:06:37.000,NYPD,Noise - Street/Sidewalk,Unspecified,Unspecified
1,70783143,2026-10-17T02:06:14.000,TLC,Taxi Complaint,In Progress,QUEENS
2,70779590,2026-10-17T02:06:13.000,NYPD,Noise - Residential,In Progress,Unspecified
3,70785024,2026-10-17T02:05:51.000,NYPD,Noise - Commercial,In Progress,QUEENS
4,70777529,2026-10-17T02:05:41.000,NYPD,Noise - Residential,In Progress,BROOKLYN


## Interpretation and next steps

- NYC 311 is a current-state feed. `status` is observed at capture, and historical status is unknown before the first snapshot.
- Source timestamps are floating NYC local times, retained as source values here. Do not label them UTC; later duration analysis needs an explicit America/New_York and DST policy.
- `due_date` is an expected agency update date, not a universal closure SLA.
- Fetching the complete date-filtered cohort requires pagination, count reconciliation and consistency checks. This notebook intentionally tests a single bounded response.
- If the API reports an unknown column (for example `descriptor_2`), inspect live metadata before changing `SELECTED_FIELDS`; this notebook does not claim live schema validation.

References: [Socrata endpoints](https://dev.socrata.com/docs/endpoints), [application tokens](https://dev.socrata.com/docs/app-tokens.html), [paging](https://dev.socrata.com/consumers/getting-started).
